In [33]:
%pip install -q mne timm einops tensorboardX h5py scikit-learn

import os, sys, re, ast, subprocess, zipfile, urllib.request, warnings
from pathlib import Path
import numpy as np, pandas as pd, mne
warnings.filterwarnings("ignore", category=FutureWarning)
mne.set_log_level("ERROR")

WORK = Path("/kaggle/working")
REPO = WORK / "LaBraM"

if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/935963004/LaBraM", str(REPO)], check=True)
sys.path.insert(0, str(REPO))
print("веса:", list((REPO / "checkpoints").glob("*")))

Note: you may need to restart the kernel to use updated packages.
веса: [PosixPath('/kaggle/working/LaBraM/checkpoints/vqnsp.pth'), PosixPath('/kaggle/working/LaBraM/checkpoints/labram-base.pth')]


In [34]:
EMO = WORK / "emotiv"; EMO.mkdir(exist_ok=True)
base = "https://zenodo.org/records/1252141/files/"
for f in ["EEGs_Guinea-Bissau.zip", "EEGs_Nigeria.zip",
          "metadata_guineabissau.csv", "metadata_nigeria.csv"]:
    if not (EMO / f).exists():
        urllib.request.urlretrieve(base + f + "?download=1", EMO / f)
    if f.endswith(".zip") and not (EMO / f[:-4]).exists():
        zipfile.ZipFile(EMO / f).extractall(EMO / f[:-4])
print("скачано")

скачано


In [35]:
CHANS = ["AF3", "F7", "F3", "FC5", "T7", "P7", "O1",
         "O2", "P8", "T8", "FC6", "F4", "F8", "AF4"]
SFREQ, WIN = 200, 2000      # 200 Гц, окна по 10 с

def to_windows(data, sfreq):
    raw = mne.io.RawArray(data * 1e-6, mne.create_info(CHANS, sfreq, "eeg"))
    raw.filter(0.1, 45.0).notch_filter(50.0).resample(SFREQ)
    x = raw.get_data() * 1e6
    n = x.shape[1] // WIN
    return x[:, : n * WIN].reshape(len(CHANS), n, WIN).transpose(1, 0, 2).astype(np.float32)

In [36]:
X, y, sub = [], [], []
for country, meta_f, folder in [("GB", "metadata_guineabissau.csv", "EEGs_Guinea-Bissau"),
                                ("NG", "metadata_nigeria.csv", "EEGs_Nigeria")]:
    meta = pd.read_csv(EMO / meta_f)
    file_col = next((c for c in meta.columns if "file" in c.lower()), None)
    for f in sorted((EMO / folder).rglob("*.csv*")):
        stem = f.name.split(".")[0]
        if file_col:
            row = meta[meta[file_col].astype(str).apply(lambda s: Path(s).name.split(".")[0] == stem)]
        else:
            nums = re.findall(r"\d+", f.name)
            if not nums: continue
            row = meta[meta["subject.id"] == int(nums[-1])]
        if row.empty:
            continue
        df = pd.read_csv(f, low_memory=False)
        cols = {c.split(".")[-1].strip().upper(): c for c in df.columns}
        if not all(c in cols for c in CHANS):
            print("мало каналов:", f.name); continue
        data = df[[cols[c] for c in CHANS]].apply(pd.to_numeric, errors="coerce").dropna().values.T
        w = to_windows(data, 128)
        label = int(str(row["Group"].iloc[0]).lower().startswith("epi"))
        X.append(w); y += [label] * len(w); sub += [f"{country}_{stem}"] * len(w)

X = np.concatenate(X); y = np.array(y); sub = np.array(sub)
country = np.array([s[:2] for s in sub])
for c in ["NG", "GB"]:
    s, l = sub[country == c], y[country == c]
    print(f"{c}: людей {len(set(s))} | эпилепсия {len(set(s[l == 1]))}, контроль {len(set(s[l == 0]))}")

NG: людей 221 | эпилепсия 128, контроль 93
GB: людей 97 | эпилепсия 51, контроль 46


In [37]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix
from modeling_finetune import labram_base_patch200_200

# список каналов LaBraM из utils.py без импорта (там нужен pyhealth)
src = (REPO / "utils.py").read_text()
standard_1020 = next(
    ast.literal_eval(n.value) for n in ast.walk(ast.parse(src))
    if isinstance(n, ast.Assign) and any(getattr(t, "id", "") == "standard_1020" for t in n.targets))
input_chans = [0] + [standard_1020.index(c) + 1 for c in CHANS]
device = torch.device("cuda")

def make_model():
    m = labram_base_patch200_200(num_classes=1, drop_path_rate=0.1, use_mean_pooling=True,
                                 init_scale=0.001, use_rel_pos_bias=False,
                                 use_abs_pos_emb=True, init_values=0.1, qkv_bias=False)
    ck = torch.load(REPO / "checkpoints" / "labram-base.pth", map_location="cpu", weights_only=False)
    ck = ck.get("model", ck)
    sd = {k[8:]: v for k, v in ck.items() if k.startswith("student.")} or ck
    sd = {k: v for k, v in sd.items() if not k.startswith("head.")}
    m.load_state_dict(sd, strict=False)
    return m.to(device)

def forward(m, xb):
    xb = xb.to(device).float() / 100
    xb = xb.reshape(xb.shape[0], xb.shape[1], -1, 200)
    return m(xb, input_chans=input_chans).squeeze(-1)

def train_epoch(m, opt, scaler, dl):
    m.train()
    for xb, yb in dl:
        with torch.autocast("cuda", dtype=torch.float16):
            loss = torch.nn.functional.binary_cross_entropy_with_logits(forward(m, xb), yb.to(device))
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()

def predict(m, Xs):
    m.eval(); out = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(Xs), 256):
            out.append(torch.sigmoid(forward(m, torch.from_numpy(Xs[i:i+256]))).float().cpu())
    return torch.cat(out).numpy()

def by_subject(p, ys, ss):
    u = np.unique(ss)
    return (np.array([p[ss == k].mean() for k in u]),
            np.array([ys[ss == k][0] for k in u]))

def report(name, ps, ys):
    pred = (ps >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(ys, pred, labels=[0, 1]).ravel()
    auc = roc_auc_score(ys, ps)
    rng = np.random.default_rng(0); boot = []
    for _ in range(2000):
        i = rng.integers(0, len(ys), len(ys))
        if len(set(ys[i])) == 2: boot.append(roc_auc_score(ys[i], ps[i]))
    print(f"\n=== {name} ===")
    print(f"людей: {len(ys)} (эпилепсия {ys.sum()}, контроль {len(ys) - ys.sum()})")
    print(f"AUC: {auc:.3f}  95% ДИ [{np.percentile(boot, 2.5):.3f}; {np.percentile(boot, 97.5):.3f}]")
    print(f"Сбалансированная точность: {balanced_accuracy_score(ys, pred):.3f}")
    print(f"Чувствительность: {tp}/{tp+fn} = {tp/max(tp+fn,1):.0%}")
    print(f"Специфичность:   {tn}/{tn+fp} = {tn/max(tn+fp,1):.0%}")

EPOCHS = 8   # по прошлому запуску качество выходит на плато к 6–8 эпохе

In [38]:
ng = np.where(country == "NG")[0]
gb = np.where(country == "GB")[0]

torch.manual_seed(0)
model = make_model()
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
scaler = torch.amp.GradScaler("cuda")
dl = DataLoader(TensorDataset(torch.from_numpy(X[ng]), torch.from_numpy(y[ng]).float()),
                batch_size=64, shuffle=True)
for ep in range(EPOCHS):
    train_epoch(model, opt, scaler, dl)
    print("эпоха", ep, "готова")

torch.save(model.state_dict(), WORK / "labram_A_nigeria.pth")
ps, ys = by_subject(predict(model, X[gb]), y[gb], sub[gb])
report("А: обучение Нигерия → проверка Гвинея-Бисау", ps, ys)

эпоха 0 готова
эпоха 1 готова
эпоха 2 готова
эпоха 3 готова
эпоха 4 готова
эпоха 5 готова
эпоха 6 готова
эпоха 7 готова

=== А: обучение Нигерия → проверка Гвинея-Бисау ===
людей: 97 (эпилепсия 51, контроль 46)
AUC: 0.711  95% ДИ [0.605; 0.807]
Сбалансированная точность: 0.695
Чувствительность: 21/51 = 41%
Специфичность:   45/46 = 98%


In [39]:
from sklearn.model_selection import StratifiedGroupKFold

all_ps, all_ys = [], []
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
for fold, (tr, te) in enumerate(cv.split(X, y, sub)):
    torch.manual_seed(fold)
    model = make_model()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
    scaler = torch.amp.GradScaler("cuda")
    dl = DataLoader(TensorDataset(torch.from_numpy(X[tr]), torch.from_numpy(y[tr]).float()),
                    batch_size=64, shuffle=True)
    for ep in range(EPOCHS):
        train_epoch(model, opt, scaler, dl)
    ps, ys = by_subject(predict(model, X[te]), y[te], sub[te])
    print(f"фолд {fold}: людей {len(ys)}, AUC = {roc_auc_score(ys, ps):.3f}")
    all_ps.append(ps); all_ys.append(ys)

report("Б: обе страны вместе, кросс-валидация по людям",
       np.concatenate(all_ps), np.concatenate(all_ys))

фолд 0: людей 64, AUC = 0.946
фолд 1: людей 63, AUC = 0.924
фолд 2: людей 64, AUC = 0.928
фолд 3: людей 64, AUC = 0.895
фолд 4: людей 63, AUC = 0.963

=== Б: обе страны вместе, кросс-валидация по людям ===
людей: 318 (эпилепсия 179, контроль 139)
AUC: 0.914  95% ДИ [0.878; 0.944]
Сбалансированная точность: 0.845
Чувствительность: 157/179 = 88%
Специфичность:   113/139 = 81%
